#PYTHON EDA FOR KPI METRICS

In [1]:
import pandas as pd 
import sqlalchemy
import pyodbc
from sqlalchemy import create_engine
import urllib

print('sqlalchemy version: ', sqlalchemy.__version__)
print(pyodbc.drivers())

sqlalchemy version:  2.0.39
['SQL Server', 'Devart ODBC Driver for SQLite', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [ ]:
# Server Credentials for pyodbc connection

server       = r'<server_name>'
driver       = 'ODBC Driver 18 for SQL Server'
database     = '<database_name>'

connection_string = (
    f"Driver={{{driver}}};"
    f"Server={server};"
    f"Database={database};"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

# Connection check
try:
    conn = pyodbc.connect(connection_string)
    conn.close()
    print("✅ Connection successful.")
except pyodbc.Error as e:
    print(f"❌ Connection failed: {e}")

In [5]:
# sqlalchemy connections

sqla_connection_string = urllib.parse.quote(connection_string)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={connection_string}")


In [6]:
# The KPIs will be defined by manufacter: HAUSBERG & WEISSTECH

first_query_hau = """SELECT 
       [source_id] 
      ,[quality]
      ,[performance]
      ,[availability]
      ,[pplh]
      ,[ftq]
      ,[oee]
  FROM [db_manufacturing_warehouse_pub].[gold_layer].[fact_status_table_final_dev]
  WHERE source_id LIKE '%HAUSBERG%'
"""
df_status_hau = pd.read_sql(first_query_hau, engine)
df_status_hau.describe()

c:\Users\amcys\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


,quality,performance,availability,pplh,ftq,oee
count,198631.000000,198631.000000,198631.000000,198631.000000,198631.000000,198631.000000
mean,0.963452,0.784195,0.882013,9.640994,0.943395,0.649877
std,0.033984,0.159126,0.120021,6.861431,0.046184,0.073557
min,0.714300,0.358400,0.457100,1.500000,0.594600,0.304300
25%,0.949200,0.636000,0.780000,4.608900,0.921300,0.600500
50%,0.973300,0.815100,0.880900,7.866700,0.956900,0.653400
75%,0.986300,0.913000,1.000000,12.333300,0.977400,0.702600
max,1.000000,1.149600,1.000000,49.920000,1.000000,0.943900


In [7]:
# The KPIs will be defined by manufacter: WEISSTECH & HAUSBERG

first_query_wei = """SELECT 
       [source_id] 
      ,[quality]
      ,[performance]
      ,[availability]
      ,[pplh]
      ,[ftq]
      ,[oee]
  FROM [db_manufacturing_warehouse_pub].[gold_layer].[fact_status_table_final_dev]
  WHERE source_id LIKE '%WEISSTECH%'
"""
df_status_wei = pd.read_sql(first_query_wei, engine)
df_status_wei.describe()

,quality,performance,availability,pplh,ftq,oee
count,198568.000000,198568.000000,198568.000000,198568.000000,198568.000000,198568.000000
mean,0.960456,0.783319,0.882343,8.117393,0.938794,0.647414
std,0.033989,0.158979,0.119818,5.318733,0.045387,0.073402
min,0.729700,0.365600,0.450700,1.422200,0.634100,0.298700
25%,0.944400,0.635600,0.780700,4.458300,0.915300,0.598200
50%,0.968300,0.814200,0.881300,6.584700,0.949000,0.651200
75%,0.984100,0.911900,1.000000,10.185200,0.972000,0.700000
max,1.000000,1.149600,1.000000,49.216600,1.000000,0.947500


In [8]:
# Checking total rows of data 
total_rows= """
SELECT
SUM(total_rows) AS total_rows_sum
FROM
    (SELECT 
          [source_id],
          COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse_pub].[gold_layer].[fact_status_table_final_dev]
    GROUP BY source_id) t
"""

df_total_rows = pd.read_sql(total_rows, engine)
df_total_rows

,total_rows_sum
0,397199


##Checking how many records we have of both WEISSTECH and HAUSBERG cars where the production surpassed 85% of oee 

In [10]:
# Total rows of HAUSBERG
total_rows_hau = """
SELECT
SUM(total_rows) AS total_rows_sum_hau
FROM
    (SELECT 
          [source_id],
          COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse_pub].[gold_layer].[fact_status_table_final_dev]
    WHERE source_id LIKE '%HAUSBERG%' AND oee >= 0.85
    GROUP BY source_id) t
"""

df_total_rows_hau = pd.read_sql(total_rows_hau, engine)
df_total_rows_hau

,total_rows_sum_hau
0,198


In [17]:
# Total rows of hau
total_rows_hau = """
SELECT
SUM(total_rows) AS total_rows_sum
FROM
    (SELECT 
          [source_id],
          COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse_pub].[gold_layer].[fact_status_table_final_dev]
    WHERE source_id LIKE '%HAUSBERG%'
    GROUP BY source_id) t
"""

# Total rows of hau products above 85% OEE
total_rows_sum_hau = """
SELECT SUM(total_rows) AS total_rows_sum
FROM (
    SELECT [source_id], COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse_pub].[gold_layer].[fact_status_table_final_dev]
    WHERE source_id LIKE '%HAUSBERG%' AND oee >= 0.85
    GROUP BY [source_id]
) t
"""

df_total_rows_hau = pd.read_sql(total_rows_hau, engine)
df_total_rows_hau_85 = pd.read_sql(total_rows_sum_hau, engine)

denom = df_total_rows_hau.iloc[0, 0]
percentage_hau = df_total_rows_hau_85.iloc[0, 0] / denom


print(f"total rows of HAUSBERG production: {df_total_rows_hau.iloc[0, 0]:,.0f}")
print(f"total rows of HAUSBERG production above 85% OEE: {df_total_rows_hau_85.iloc[0, 0]:,.0f}")
print(f"% of rows with more than 85% oee: {percentage_hau:.2%}")

total rows of HAUSBERG production: 198,631
total rows of HAUSBERG production above 85% OEE: 198
% of rows with more than 85% oee: 0.10%


In [20]:
# Total rows of WEISSTECH
total_rows_wei = """
SELECT SUM(total_rows) AS total_rows_sum
FROM (
    SELECT [source_id], COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse_pub].[gold_layer].[fact_status_table_final_dev]
    WHERE source_id LIKE '%WEISSTECH%'
    GROUP BY source_id
) t
"""

# Total rows of WEISSTECH above 85% OEE
total_rows_sum_wei = """
SELECT SUM(total_rows) AS total_rows_sum
FROM (
    SELECT [source_id], [oee], COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse_pub].[gold_layer].[fact_status_table_final_dev]
    WHERE source_id LIKE '%WEISSTECH%' AND [oee] > 0.85
    GROUP BY source_id, oee
) t
"""

df_total_rows_wei = pd.read_sql(total_rows_wei, engine)
df_total_rows_wei_85 = pd.read_sql(total_rows_sum_wei, engine)

denom = df_total_rows_wei.iloc[0, 0]
percentage_wei = df_total_rows_wei_85.iloc[0, 0] / denom if denom else 0

print(f"total rows of WEISSTECH production: {df_total_rows_wei.iloc[0, 0]:,.0f}")
print(f"total rows of WEISSTECH production above 85% OEE: {df_total_rows_wei_85.iloc[0, 0]:,.0f}")
print(f"% of rows with more than 85% oee: {percentage_wei:.2%}")

total rows of WEISSTECH production: 198,568
total rows of WEISSTECH production above 85% OEE: 179
% of rows with more than 85% oee: 0.09%


## Final Conclusion

> ⚠️ The cell outputs above are stale: they were captured before the dataset reload and before PPLH was redefined as parts per operator-hour (previously the formula was missing the minutes→hours conversion, which is why the old PPLH values sat below 1). The figures below come from the current `gold_layer.fact_status_table_final_dev`. Re-run the notebook to refresh the outputs.

In HAUSBERG products, only **198** production records got an OEE higher than **85%**, which corresponds to **0.10%** of all HAUSBERG status data (198,631 rows).

In WEISSTECH products, only **181** production records got an OEE higher than **85%**, which corresponds to **0.09%** of all WEISSTECH status data (198,568 rows).

Since we have too few records to use the golden OEE rule of 85%, we'll base ourselves on the third quartile (75%).

### HAUSBERG

The HAUSBERG KPI goals for green color 🟢 will be:

- OEE: **70.26%** *(75th percentile)*
- Performance: **90.0%**
- Availability: **78.86%** *(derived: OEE ÷ (Performance × Quality))*
- Quality: **99.0%**
- FTQ: **97.74%** *(75th percentile)*
- PPLH: **12.33** *(75th percentile)*

The HAUSBERG KPI goals for yellow color 🟡 are derived from the green goals — OEE and FTQ at 95% of green, PPLH at 90% of green, Performance and Quality relaxed slightly, and Availability recalculated as OEE ÷ (Performance × Quality):

- OEE: **66.75%**
- Performance: **88.0%**
- Availability: **77.40%**
- Quality: **98.0%**
- FTQ: **92.85%**
- PPLH: **11.10**

Below the minimum for the yellow KPIs, we'll display the color red 🔴.

Sometimes the performance surpasses 100% and this is incorrect. To call the user's attention we will display it with purple 🟣.

Values of reference (HAUSBERG, 198,631 rows):

|       | quality | performance | availability | pplh | ftq | oee |
|-------|--------:|------------:|-------------:|-----:|----:|----:|
| mean  | 0.963451 | 0.784194 | 0.882012 | 9.640994 | 0.943395 | 0.649877 |
| std   | 0.033984 | 0.159126 | 0.120021 | 6.861431 | 0.046184 | 0.073557 |
| min   | 0.714300 | 0.358400 | 0.457100 | 1.500000 | 0.594600 | 0.304300 |
| 25%   | 0.949200 | 0.636000 | 0.780000 | 4.608900 | 0.921300 | 0.600500 |
| 50%   | 0.973300 | 0.815100 | 0.880900 | 7.866700 | 0.956900 | 0.653400 |
| 75%   | 0.986300 | 0.913000 | 1.000000 | 12.333300 | 0.977400 | 0.702600 |
| max   | 1.000000 | 1.149600 | 1.000000 | 49.920000 | 1.000000 | 0.943900 |

### WEISSTECH

The WEISSTECH KPI goals for green color 🟢 will be:

- OEE: **70.00%** *(75th percentile)*
- Performance: **90.0%**
- Availability: **78.56%** *(derived: OEE ÷ (Performance × Quality))*
- Quality: **99.0%**
- FTQ: **97.20%** *(75th percentile)*
- PPLH: **10.18** *(75th percentile)*

The WEISSTECH KPI goals for yellow color 🟡 follow the same derivation as HAUSBERG:

- OEE: **66.50%**
- Performance: **88.0%**
- Availability: **77.11%**
- Quality: **98.0%**
- FTQ: **92.34%**
- PPLH: **9.16**

Below the minimum for the yellow KPIs, we'll display the color red 🔴.

Sometimes the performance surpasses 100% and this is incorrect. To call the user's attention we will display it with purple 🟣.

Values of reference (WEISSTECH, 198,568 rows):

|       | quality | performance | availability | pplh | ftq | oee |
|-------|--------:|------------:|-------------:|-----:|----:|----:|
| mean  | 0.960456 | 0.783319 | 0.882343 | 8.117393 | 0.938794 | 0.647413 |
| std   | 0.033989 | 0.158979 | 0.119818 | 5.318733 | 0.045387 | 0.073402 |
| min   | 0.729700 | 0.365600 | 0.450700 | 1.422200 | 0.634100 | 0.298700 |
| 25%   | 0.944400 | 0.635600 | 0.780700 | 4.458300 | 0.915300 | 0.598200 |
| 50%   | 0.968300 | 0.814200 | 0.881300 | 6.584700 | 0.949000 | 0.651200 |
| 75%   | 0.984100 | 0.911900 | 1.000000 | 10.185200 | 0.972000 | 0.700000 |
| max   | 1.000000 | 1.149600 | 1.000000 | 49.216600 | 1.000000 | 0.947500 |